# DSCR Screening Across a Market

**Goal:** Use Homecastr's proforma metrics to screen neighborhoods for debt-service coverage and filter to lender-qualifying DSCR ≥ 1.20.

Institutional lenders typically require DSCR ≥ 1.20 for investment property loans. This notebook shows how to:

1. Pull neighborhood proforma data across Houston using H3 hex cells
2. Filter to neighborhoods meeting DSCR thresholds
3. Rank survivors by risk-adjusted growth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/investment/01_dscr_screening.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

# Lender thresholds
DSCR_MIN = 1.20
RELIABILITY_MIN = 0.70   # discard low-confidence predictions
CAP_RATE_MIN = 0.045     # 4.5% minimum cap rate

In [ ]:
# Sample of H3 resolution-8 hexes covering central Houston.
# In production: generate these programmatically with h3.grid_disk() or
# h3.polygon_to_cells() from a shapefile boundary.

houston_hexes = [
    "882a100c65fffff",  # Midtown
    "882a1008b5fffff",  # Heights
    "882a100d25fffff",  # Montrose
    "882a100255fffff",  # EaDo
    "882a100275fffff",  # Museum District
    "882a100695fffff",  # Bellaire
    "882a1006b5fffff",  # West U
    "882a100eb5fffff",  # River Oaks
    "882a100ee5fffff",  # Upper Kirby
    "882a100ef5fffff",  # Rice Military
]

print(f"Fetching proforma data for {len(houston_hexes)} Houston neighborhoods...")
df = client.forecast.by_hex.retrieve_many(houston_hexes, year=2028)
print(f"Retrieved {len(df)} neighborhoods.")
df.head()

In [ ]:
# Screen by lender thresholds
qualified = df[
    (df["dscr"] >= DSCR_MIN) &
    (df["reliability"] >= RELIABILITY_MIN) &
    (df["cap_rate"] >= CAP_RATE_MIN)
].copy()

print(f"{len(qualified)} / {len(df)} neighborhoods pass all three thresholds.")
print(f"  DSCR ≥ {DSCR_MIN}")
print(f"  Reliability ≥ {RELIABILITY_MIN}")
print(f"  Cap Rate ≥ {CAP_RATE_MIN:.1%}")

# Risk-adjusted ranking
qualified["risk_adj_growth"] = qualified["opportunity"] * qualified["reliability"]
qualified = qualified.sort_values("risk_adj_growth", ascending=False)

cols = ["location", "dscr", "cap_rate", "monthly_rent", "opportunity", "reliability", "risk_adj_growth"]
print("\nQualified neighborhoods (ranked by risk-adjusted growth):")
print(qualified[cols].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# All neighborhoods — DSCR distribution
axes[0].barh(df["location"].fillna(df["h3_id"]),
             df["dscr"],
             color=["steelblue" if d >= DSCR_MIN else "lightcoral" for d in df["dscr"]])
axes[0].axvline(DSCR_MIN, color="red", linestyle="--", linewidth=1.5, label=f"Min DSCR {DSCR_MIN}")
axes[0].set_xlabel("DSCR")
axes[0].set_title("DSCR by Neighborhood")
axes[0].legend()

# Qualified — risk-adjusted growth
if len(qualified) > 0:
    axes[1].barh(qualified["location"].fillna(qualified["h3_id"]),
                 qualified["risk_adj_growth"], color="steelblue")
    axes[1].set_xlabel("Risk-Adjusted Growth (opportunity × reliability)")
    axes[1].set_title(f"Qualified Neighborhoods (DSCR ≥ {DSCR_MIN})")
else:
    axes[1].text(0.5, 0.5, "No neighborhoods passed all thresholds",
                 ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
qualified.to_csv("houston_dscr_qualified.csv", index=False)
print(f"Saved {len(qualified)} qualified neighborhoods to houston_dscr_qualified.csv")